In [1]:
!pip install -q "datasets<3.0" transformers peft trl accelerate bitsandbytes wandb torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [3]:
from huggingface_hub import login
login()

In [5]:

import wandb
wandb.login()

True

In [6]:
from google.colab import files
uploaded = files.upload()  # upload train.jsonl and validation.jsonl

Saving validation.jsonl to validation.jsonl
Saving train.jsonl to train.jsonl


In [7]:
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("train.jsonl")
val_data = load_jsonl("validation.jsonl")

print(f"Train examples: {len(train_data):,}")
print(f"Val examples:   {len(val_data):,}")
print(f"\nSample training example:")
print(train_data[0]['text'])

Train examples: 56,355
Val examples:   8,421

Sample training example:
### Input:
Columns: State/territory (text), Text/background colour (text), Format (text), Current slogan (text), Current series (text), Notes (text)

Question: Tell me what the notes are for South Australia 

### SQL:
SELECT `Notes` FROM table WHERE `Current slogan` = 'SOUTH AUSTRALIA'


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"Done! Memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading tokenizer...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Done! Memory used: 6.1 GB


In [9]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Prepare model for training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [13]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Convert to HuggingFace datasets
train_dataset = Dataset.from_list([{"text": ex["text"]} for ex in train_data])
val_dataset = Dataset.from_list([{"text": ex["text"]} for ex in val_data])

training_args = SFTConfig(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,        # effective batch size = 16
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name="llama3-wikisql-qlora-a100",
    max_seq_length=256,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("Trainer ready!")
print(f"Total training steps: {len(trainer.get_train_dataloader()) * training_args.num_train_epochs}")

Converting train dataset to ChatML:   0%|          | 0/56355 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/56355 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/56355 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/56355 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/8421 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/8421 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8421 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/8421 [00:00<?, ? examples/s]

Trainer ready!
Total training steps: 10569


In [14]:
import wandb

wandb.init(project="text-to-sql-llama", name="llama3-wikisql-qlora")

print("Starting training...")
trainer.train()

print("Training complete!")

train/epoch,▁▂▃▄▆▇█
train/global_step,▁▂▃▅▆▇█
train/grad_norm,█▄▃▂▃▁▃
train/learning_rate,█▇▆▅▃▂▁
train/loss,█▂▂▁▂▁▁
train/mean_token_accuracy,▁▆▇█▇██
train/epoch,0.01987
train/global_step,70
train/grad_norm,0.85674
train/learning_rate,0.0002
train/loss,0.82894


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.497181,0.620473
2,0.403196,0.630884
3,0.340801,0.678062


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training complete!


In [23]:
# Save the LoRA adapter locally
model.save_pretrained("./llama3-wikisql-qlora")
tokenizer.save_pretrained("./llama3-wikisql-qlora")
print("Saved locally!")

Saved locally!


In [25]:
from huggingface_hub import login
login(token=".....")

model.push_to_hub(".....")
tokenizer.push_to_hub(".....")
print("Pushed to Hub!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   3%|3         | 5.54MB /  168MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpyw_k7x5q/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Pushed to Hub!


In [26]:
import json

training_results = {
    "model": "meta-llama/Meta-Llama-3-8B-Instruct",
    "adapter": "oLittle-five/llama3-8b-wikisql-qlora",
    "dataset": "wikisql",
    "epochs": 3,
    "best_epoch": 1,
    "results": [
        {"epoch": 1, "train_loss": 0.497181, "val_loss": 0.620473},
        {"epoch": 2, "train_loss": 0.403196, "val_loss": 0.630884},
        {"epoch": 3, "train_loss": 0.340801, "val_loss": 0.678062},
    ],
    "hyperparameters": {
        "lora_r": 16,
        "lora_alpha": 32,
        "learning_rate": 2e-4,
        "batch_size": 16,
        "gradient_accumulation_steps": 2,
        "max_seq_length": 256,
    }
}

with open("training_results.json", "w") as f:
    json.dump(training_results, f, indent=2)

print("Saved!")

Saved!
